# Step 06.1 — Vocabulary 与 Character Tokenizer

神经网络不能直接把字符串 `"hello"` 当成矩阵计算。

所以语言模型的第一步是：

```text
text
  ↓ tokenizer
token IDs
```

这一节先使用最透明的 **character-level tokenizer**：一个字符就是一个 token。

目标：理解 vocabulary、token ID、`stoi`、`itos`、`encode`、`decode`。

## 1. 为什么文字必须先变成整数？

假设文本是：

```text
hello
```

模型后面真正接收的不会是字母 `h/e/l/o`，而会是整数 ID，例如：

```text
h -> 1
e -> 0
l -> 2
o -> 3

"hello" -> [1, 0, 2, 2, 3]
```

非常重要：这些数字只是 **category labels（类别编号）**。

`o -> 3` 并不表示 `o` 比 `e -> 0` “大三倍”、更重要或更相似。

真正有语义的连续向量表示，要到 Step 7 的 Embedding 才会学习。

## 2. Vocabulary：模型允许出现哪些 token？

先用一小段文本：

```text
hello lulu\n
```

Vocabulary（词表）就是这段语料里所有**不同 token 的集合**。

Character tokenizer 下，一个 token 就是一个字符，因此空格和换行也是真正的 token。

In [ ]:
text = "hello lulu\n"

chars = sorted(set(text))
vocab_size = len(chars)

print("text       :", repr(text))
print("vocabulary :", chars)
print("vocab_size :", vocab_size)

`set(text)` 去掉重复字符；`sorted(...)` 只是让编号顺序稳定、便于复现。

例如可能得到：

```text
['\n', ' ', 'e', 'h', 'l', 'o', 'u']
```

于是：

```text
vocab_size = 7
```

这里的 `vocab_size` 会在以后直接决定模型最后要为多少种 token 产生预测分数。

## 3. `stoi` 与 `itos`：建立双向查表

`stoi` = string-to-integer：

```text
字符 -> token ID
```

`itos` = integer-to-string：

```text
token ID -> 字符
```

我们用 Python dictionary 建立这两个映射。

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print("stoi:", stoi)
print("itos:", itos)

如果：

```text
'h' -> 3
```

那么反向必须有：

```text
3 -> 'h'
```

这样才能把模型使用的 ID 再还原成人能读的文字。

## 4. `encode` 与 `decode`

Tokenizer 最基本的两个动作就是：

```text
encode: text -> IDs
decode: IDs  -> text
```

In [ ]:
def encode(s: str) -> list[int]:
    return [stoi[ch] for ch in s]


def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)


message = "hello"
encoded = encode(message)
decoded = decode(encoded)

print("original:", message)
print("encoded :", encoded)
print("decoded :", decoded)
print("round trip correct:", decoded == message)

这里最好形成一个非常清楚的 round trip：

```text
"hello"
   ↓ encode
[3, 2, 4, 4, 5]
   ↓ decode
"hello"
```

具体 ID 取决于 vocabulary 排序，但 encode 后再 decode 应该恢复原字符串。

In [ ]:
import mlx.core as mx

data = mx.array(encode(text))

print("token IDs:", data)
print("shape    :", data.shape)
print("dtype    :", data.dtype)

现在文本第一次真正变成 MLX tensor：

```text
text
↓ tokenizer
[token_id_0, token_id_1, ...]
↓ mx.array
integer tensor
```

后面的模型就从这些整数 token IDs 开始工作。

## 5. Character tokenizer 的优点、局限与下一步

Character-level tokenizer 的优点：

- 极其简单；
- vocabulary 很容易看懂；
- encode/decode 完全透明；
- 非常适合从零教学 MiniGPT。

局限也很明显：

- 一个单词会被拆成很多 token；
- sequence 会更长；
- 不像现代 LLM 常用的 BPE / byte-level subword tokenizer 那样高效。

还有一个重要边界：如果 `encode()` 遇到 vocabulary 中不存在的新字符，当前简单实现会报错。真正 tokenizer 会设计 unknown/byte fallback 等机制；本项目核心教学阶段先使用固定语料词表，因此暂时不扩展。

### 6.1 最终结论

- tokenizer 把离散文字转换为 token IDs；
- vocabulary 是所有允许 token 的集合；
- `vocab_size` 是 token 类别总数；
- token ID 只是类别编号，本身没有连续几何语义；
- `stoi` / `itos` 提供双向映射；
- `encode` / `decode` 完成文字与 ID 序列之间的转换；
- 下一节 6.2：为什么不能直接把这些整数 ID 当成连续数值输入神经网络，以及 One-Hot 与 learned Embedding 的区别。